# Camada Bronze

Objetivo: trazer os arquivos da competição para o schema `bronze` do banco `data/pokemon.duckdb` **sem alterar nenhum valor**.

| Arquivo em `data/` | Conteúdo | Tabela |
|---|---|---|
| `EN_Card_Data_R2_full.csv` | Cartas em inglês, uma linha por ataque ou habilidade | `bronze.cartas_en` |
| `JP_Card_Data_R2_full.csv` | As mesmas cartas em japonês | `bronze.cartas_jp` |
| `deck_exemplo.csv` | Deck de 60 cartas do exemplo de submissão (só os IDs, sem cabeçalho) | `bronze.deck_exemplo` |

Regras desta camada e o motivo de cada uma:
1. Todas as colunas são lidas como texto, inclusive números. Assim nenhum valor é convertido ou arredondado na leitura.
2. Textos como `n/a` e campos vazios ficam como estão. A decisão sobre o que é nulo é tomada na silver, onde fica documentada.
3. Os nomes das colunas são os originais, em inglês e em japonês.
4. Uma tabela de controle registra linhas, colunas e o hash MD5 de cada arquivo, para conferir que a carga é fiel à origem.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd

from pipeline import config
from pipeline.bronze import carregar_bronze
from pipeline.db import conectar, ler_tabela, listar_tabelas, salvar_tabela

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

## 1. Leitura das fontes

In [2]:
fontes = {
    "cartas_en": {"caminho": config.ARQUIVO_CARTAS_EN},
    "cartas_jp": {"caminho": config.ARQUIVO_CARTAS_JP},
    "deck_exemplo": {"caminho": config.ARQUIVO_DECK, "cabecalho": False, "nomes": ["Card ID"]},
}

tabelas = carregar_bronze(fontes)
tabelas["controle_carga"]

,tabela,arquivo_origem,linhas,colunas,md5_arquivo,carregado_em
0,cartas_en,EN_Card_Data_R2_full.csv,2273,18,07f05430d09423e2972c425785812008,2026-11-08T19:54:27+00:00
1,cartas_jp,JP_Card_Data_R2_full.csv,2273,17,bf3ac85d0a82732e96f993fc50fa9e2d,2026-11-08T19:54:27+00:00
2,deck_exemplo,deck_exemplo.csv,60,1,f196fa73c10efc146fb0c3c553c2eb3b,2026-11-08T19:54:27+00:00


## 2. Primeira olhada nos dados brutos

O arquivo em inglês tem uma linha por ataque ou habilidade. Uma carta com dois ataques aparece em duas linhas
repetindo os dados da carta (HP, tipo, fraqueza etc.). Esse é o principal ajuste de modelagem da camada silver.

In [3]:
cartas_en = tabelas["cartas_en"]
cartas_en.head(8)

,Card ID,Card Name,Expansion,Collection No.,Stage (Pokémon)/Type (Energy and Trainer),Rule,Category,Previous stage,HP,Type,Weakness,Resistance (Type),Retreat,Move Name,Cost,Damage,Effect Explanation,Species
0,1,Basic {G} Energy,SVE,1,Basic Energy,n/a,n/a,n/a,n/a,{G},,,n/a,,n/a,n/a,,n/a
1,2,Basic {R} Energy,SVE,2,Basic Energy,n/a,n/a,n/a,n/a,{R},,,n/a,,n/a,n/a,,n/a
2,3,Basic {W} Energy,SVE,3,Basic Energy,n/a,n/a,n/a,n/a,{W},,,n/a,,n/a,n/a,,n/a
3,4,Basic {L} Energy,SVE,4,Basic Energy,n/a,n/a,n/a,n/a,{L},,,n/a,,n/a,n/a,,n/a
4,5,Basic {P} Energy,SVE,5,Basic Energy,n/a,n/a,n/a,n/a,{P},,,n/a,,n/a,n/a,,n/a
5,6,Basic {F} Energy,SVE,6,Basic Energy,n/a,n/a,n/a,n/a,{F},,,n/a,,n/a,n/a,,n/a
6,7,Basic {D} Energy,SVE,7,Basic Energy,n/a,n/a,n/a,n/a,{D},,,n/a,,n/a,n/a,,n/a
7,8,Basic {M} Energy,SVE,8,Basic Energy,n/a,n/a,n/a,n/a,{M},,,n/a,,n/a,n/a,,n/a


In [4]:
cartas_jp = tabelas["cartas_jp"]
cartas_jp.head(3)

,カード ID,カード名,エキスパンションマーク,コレクション番号,ポケモンの進化の段階/エネルギー・トレーナーズの種類,ルール,カテゴリ,進化前,HP,タイプ,弱点,抵抗力,にげる,ワザ名,コスト,ダメージ,効果の説明
0,1,基本【草】エネルギー,,GRA,基本エネルギー,,,,,草,,,,,,,
1,2,基本【炎】エネルギー,,FIR,基本エネルギー,,,,,炎,,,,,,,
2,3,基本【水】エネルギー,,WAT,基本エネルギー,,,,,水,,,,,,,


In [5]:
print("Colunas EN:", len(cartas_en.columns), "| Colunas JP:", len(cartas_jp.columns))
print("Cartas distintas EN:", cartas_en["Card ID"].nunique())
print("Cartas distintas JP:", cartas_jp["カード ID"].nunique())
print("Linhas do deck exemplo:", len(tabelas["deck_exemplo"]))

Colunas EN: 18 | Colunas JP: 17
Cartas distintas EN: 1431
Cartas distintas JP: 1431
Linhas do deck exemplo: 60


## 3. Perfil de valores ausentes no dado bruto

No arquivo em inglês os ausentes aparecem de duas formas: o texto `n/a` (campo que não se aplica, como HP de uma carta
de Treinador) e o campo vazio. Esse levantamento orienta a limpeza da camada silver.

In [6]:
perfil = pd.DataFrame({
    "qtd_n/a": (cartas_en == "n/a").sum(),
    "qtd_vazio": (cartas_en == "").sum(),
})
perfil["pct_ausente"] = ((perfil["qtd_n/a"] + perfil["qtd_vazio"]) / len(cartas_en) * 100).round(1)
perfil.sort_values("pct_ausente", ascending=False)

,qtd_n/a,qtd_vazio,pct_ausente
Species,2067,0,90.9
Rule,1889,0,83.1
Category,1877,0,82.6
Resistance (Type),1590,240,80.5
Previous stage,1302,0,57.3
Damage,826,0,36.3
Effect Explanation,594,8,26.5
Cost,518,0,22.8
Weakness,80,240,14.1
Retreat,304,0,13.4


## 4. Gravação no schema bronze

In [7]:
conexao = conectar(config.BANCO)
for nome, dados in tabelas.items():
    salvar_tabela(conexao, "bronze", nome, dados)
listar_tabelas(conexao, "bronze")

['cartas_en', 'cartas_jp', 'controle_carga', 'deck_exemplo']

## 5. Conferência: o banco guarda exatamente o que foi lido

In [8]:
for nome in fontes:
    gravado = ler_tabela(conexao, "bronze", nome)
    pd.testing.assert_frame_equal(gravado, tabelas[nome])
    print(f"bronze.{nome}: {len(gravado)} linhas conferidas")
conexao.close()

bronze.cartas_en: 2273 linhas conferidas
bronze.cartas_jp: 2273 linhas conferidas
bronze.deck_exemplo: 60 linhas conferidas
